In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.tree import export_text
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score , accuracy_score , precision_score , recall_score , f1_score , fbeta_score
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.model_selection import ParameterGrid
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

**EDA**

In [3]:
data.shape

(100000, 31)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 31 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   age                                 100000 non-null  int64  
 1   gender                              100000 non-null  object 
 2   ethnicity                           100000 non-null  object 
 3   education_level                     100000 non-null  object 
 4   income_level                        100000 non-null  object 
 5   employment_status                   100000 non-null  object 
 6   smoking_status                      100000 non-null  object 
 7   alcohol_consumption_per_week        100000 non-null  int64  
 8   physical_activity_minutes_per_week  100000 non-null  int64  
 9   diet_score                          100000 non-null  float64
 10  sleep_hours_per_day                 100000 non-null  float64
 11  screen_time_hours_per_day  

In [5]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
age,100000.0,50.120410,15.604600,18.00,39.00,50.00,61.00,90.00
alcohol_consumption_per_week,100000.0,2.003670,1.417779,0.00,1.00,2.00,3.00,10.00
physical_activity_minutes_per_week,100000.0,118.911640,84.409662,0.00,57.00,100.00,160.00,833.00
diet_score,100000.0,5.994787,1.780954,0.00,4.80,6.00,7.20,10.00
sleep_hours_per_day,100000.0,6.997818,1.094622,3.00,6.30,7.00,7.70,10.00
screen_time_hours_per_day,100000.0,5.996468,2.468406,0.50,4.30,6.00,7.70,16.80
family_history_diabetes,100000.0,0.219410,0.413849,0.00,0.00,0.00,0.00,1.00
hypertension_history,100000.0,0.250800,0.433476,0.00,0.00,0.00,1.00,1.00
cardiovascular_history,100000.0,0.079200,0.270052,0.00,0.00,0.00,0.00,1.00
bmi,100000.0,25.612653,3.586705,15.00,23.20,25.60,28.00,39.20


**We will remove the following columns because they do not add any significant information to the model.**

In [6]:
data.drop(columns=['ethnicity', 'education_level','income_level','employment_status'], inplace=True)

In [7]:
data.head().T

,0,1,2,3,4
age,58,48,60,74,46
gender,Male,Female,Male,Female,Male
smoking_status,Never,Former,Never,Never,Never
alcohol_consumption_per_week,0,1,1,0,1
physical_activity_minutes_per_week,215,143,57,49,109
diet_score,5.7,6.7,6.4,3.4,7.2
sleep_hours_per_day,7.9,6.5,10.0,6.6,7.4
screen_time_hours_per_day,7.9,8.7,8.1,5.2,5.0
family_history_diabetes,0,0,1,0,0
hypertension_history,0,0,0,0,0


In [8]:
data.isnull().sum()

,0
age,0
gender,0
smoking_status,0
alcohol_consumption_per_week,0
physical_activity_minutes_per_week,0
diet_score,0
sleep_hours_per_day,0
screen_time_hours_per_day,0
family_history_diabetes,0
hypertension_history,0


In [9]:
data.dtypes

,0
age,int64
gender,object
smoking_status,object
alcohol_consumption_per_week,int64
physical_activity_minutes_per_week,int64
diet_score,float64
sleep_hours_per_day,float64
screen_time_hours_per_day,float64
family_history_diabetes,int64
hypertension_history,int64


In [10]:
Categorical_Col = list(data.dtypes[data.dtypes == 'object'].index)

In [11]:
for col in Categorical_Col :
  data[col] = data[col].str.lower().str.replace(' ','_')

In [12]:
Categorical_Col

['gender', 'smoking_status', 'diabetes_stage']

**We will now examine the data we have.**


In [13]:
for col in data.columns:
    print(f'Column Name: {col}')
    print('First 5 Values:')
    print(data[col].head())
    print(f'Number Of Unique Values Is : {data[col].nunique()}')
    print(f'Unique Values Is : {data[col].unique()}')
    print(f'Duplicates Within Column: {data[col].duplicated().sum()}')
    print(f'Missing Values (NaN): {data[col].isnull().sum()}')

    if pd.api.types.is_numeric_dtype(data[col]) and data[col].nunique() > 2:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers_count = ((data[col] < lower_bound) | (data[col] > upper_bound)).sum()
        print(f'Outliers count (IQR method): {outliers_count}')
    else:
        print('Outliers count: Not applicable (Categorical or Binary column)')

    print('---------------------------------------------------------------------\n')

Column Name: age
First 5 Values:
0    58
1    48
2    60
3    74
4    46
Name: age, dtype: int64
Number Of Unique Values Is : 73
Unique Values Is : [58 48 60 74 46 75 62 42 59 43 54 19 22 41 34 55 35 27 73 51 52 32 56 40
 45 80 50 33 63 30 53 29 26 38 67 44 39 66 65 37 47 31 72 49 18 89 68 28
 85 25 71 24 61 69 57 36 90 64 87 20 84 78 70 76 83 79 23 81 77 21 86 82
 88]
Duplicates Within Column: 99927
Missing Values (NaN): 0
Outliers count (IQR method): 0
---------------------------------------------------------------------

Column Name: gender
First 5 Values:
0      male
1    female
2      male
3    female
4      male
Name: gender, dtype: object
Number Of Unique Values Is : 3
Unique Values Is : ['male' 'female' 'other']
Duplicates Within Column: 99997
Missing Values (NaN): 0
Outliers count: Not applicable (Categorical or Binary column)
---------------------------------------------------------------------

Column Name: smoking_status
First 5 Values:
0     never
1    former
2     never
3

In [14]:
data.shape

(100000, 27)

Since I will be using the Decision Tree Family, the presence of outliers will not affect the model's accuracy.

**Now I will split the data, in addition to using `stratify` to ensure that the distribution ratio of patients and healthy individuals remains constant and consistent across all datasets (Train, Validation, and Test) at the same ratio found in the original data.**


*   Train Data = 60 %
*   Validation Data = 20 %
*   Test Data = 20 %

In [15]:
Train_Val_Data , Test_Data = train_test_split(data , test_size = 0.2 , random_state = 1 , stratify=data['diagnosed_diabetes'])
Train_Data , Val_Data  = train_test_split(Train_Val_Data , test_size = 0.25 , random_state = 1 , stratify=Train_Val_Data['diagnosed_diabetes'])

In [16]:
Train_Data = Train_Data.reset_index(drop=True)
Val_Data = Val_Data.reset_index(drop=True)
Test_Data = Test_Data.reset_index(drop=True)
Train_Val_Data = Train_Val_Data.reset_index(drop=True)

In [17]:
y_Train = Train_Data['diagnosed_diabetes'].values
y_Val = Val_Data['diagnosed_diabetes'].values
y_Test = Test_Data['diagnosed_diabetes'].values

In [18]:
# The diabetes_stage column was removed to prevent data leakage because it contains values that indicate
# the data specific to the diagnosed_diabetes column.

comparison_data = data[['diagnosed_diabetes', 'diabetes_stage']].head(10)
display(comparison_data)

,diagnosed_diabetes,diabetes_stage
0,1,type_2
1,0,no_diabetes
2,1,type_2
3,1,type_2
4,1,type_2
5,0,pre-diabetes
6,0,pre-diabetes
7,1,type_2
8,1,type_2
9,0,no_diabetes


In [19]:
cols_to_drop = ['diagnosed_diabetes', 'diabetes_stage']

Train_Data.drop(columns=cols_to_drop, inplace=True)
Val_Data.drop(columns=cols_to_drop, inplace=True)
Test_Data.drop(columns=cols_to_drop, inplace=True)

In [20]:
# The following columns are binary categorical columns, except for the alcohol_consumption_per_week column,
# which contains values with a maximum of 10.
# The columns ['cardiovascular_history', 'hypertension_history', and 'family_history_diabetes']
# were added to the Object_col variable because these columns are binary categorical variables
# and not numerical columns.

Categorical_Col = Categorical_Col + ['cardiovascular_history','hypertension_history','family_history_diabetes']

In [21]:
if 'diabetes_stage' in Categorical_Col:
    Categorical_Col.remove('diabetes_stage')

Categorical_Col

['gender',
 'smoking_status',
 'cardiovascular_history',
 'hypertension_history',
 'family_history_diabetes']

In [22]:
Numeric_col = list(data.dtypes[data.dtypes != 'object'].index)
Numeric_col

['age',
 'alcohol_consumption_per_week',
 'physical_activity_minutes_per_week',
 'diet_score',
 'sleep_hours_per_day',
 'screen_time_hours_per_day',
 'family_history_diabetes',
 'hypertension_history',
 'cardiovascular_history',
 'bmi',
 'waist_to_hip_ratio',
 'systolic_bp',
 'diastolic_bp',
 'heart_rate',
 'cholesterol_total',
 'hdl_cholesterol',
 'ldl_cholesterol',
 'triglycerides',
 'glucose_fasting',
 'glucose_postprandial',
 'insulin_level',
 'hba1c',
 'diabetes_risk_score',
 'diagnosed_diabetes']

In [23]:
# The following columns 'cardiovascular_history', 'hypertension_history', and 'family_history_diabetes'
# were removed from the Numeric_col variable because these columns are fundamentally binary categorical columns
# and not numerical, in addition to removing the diagnosed_diabetes column because,
# as we said, it is due to data leakage.

to_remove = ['cardiovascular_history', 'hypertension_history', 'family_history_diabetes', 'diagnosed_diabetes']
Numeric_col = [col for col in Numeric_col if col not in to_remove]
Numeric_col

['age',
 'alcohol_consumption_per_week',
 'physical_activity_minutes_per_week',
 'diet_score',
 'sleep_hours_per_day',
 'screen_time_hours_per_day',
 'bmi',
 'waist_to_hip_ratio',
 'systolic_bp',
 'diastolic_bp',
 'heart_rate',
 'cholesterol_total',
 'hdl_cholesterol',
 'ldl_cholesterol',
 'triglycerides',
 'glucose_fasting',
 'glucose_postprandial',
 'insulin_level',
 'hba1c',
 'diabetes_risk_score']

In [24]:
#################################################

In [25]:
Train_Val_Data.head()

,age,gender,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,family_history_diabetes,hypertension_history,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,67,female,never,5,136,8.1,7.6,9.3,1,0,...,57,112,204,103,149,10.12,6.21,46.8,pre-diabetes,0
1,49,female,former,2,48,5.7,8.1,9.6,0,0,...,46,115,132,115,237,10.73,8.18,31.3,type_2,1
2,29,male,current,5,72,3.8,8.3,6.3,0,0,...,40,50,80,103,115,25.03,5.79,24.7,pre-diabetes,0
3,54,male,current,1,70,7.5,6.5,6.4,1,0,...,60,98,86,137,202,5.07,7.73,43.6,type_2,1
4,73,male,current,0,29,3.3,6.9,11.2,0,1,...,49,118,99,131,202,16.63,7.46,40.2,type_2,1


In [26]:
The_Overall_Infection_Rate = Train_Val_Data['diagnosed_diabetes'].mean()
The_Overall_Infection_Rate

# The Overall Infection Rate Equal 60 %

np.float64(0.599975)

**To determine the effect of the Categorical columns on the target column.**

In [27]:
from IPython.display import display

for c in Categorical_Col:
    print(c)
    df_group = Train_Val_Data.groupby(c).diagnosed_diabetes.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - The_Overall_Infection_Rate
    df_group['risk'] = df_group['mean'] / The_Overall_Infection_Rate
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.599467,40174,-0.000508,0.999154
male,0.599430,38228,-0.000545,0.999091
other,0.625782,1598,0.025807,1.043014




smoking_status


,mean,count,diff,risk
smoking_status,,,,
current,0.601356,16225,0.001381,1.002302
former,0.601790,15866,0.001815,1.003025
never,0.598906,47909,-0.001069,0.998219




cardiovascular_history


,mean,count,diff,risk
cardiovascular_history,,,,
0,0.595789,73667,-0.004186,0.993023
1,0.648666,6333,0.048691,1.081155




hypertension_history


,mean,count,diff,risk
hypertension_history,,,,
0,0.591584,59907,-0.008391,0.986014
1,0.624994,20093,0.025019,1.041700




family_history_diabetes


,mean,count,diff,risk
family_history_diabetes,,,,
0,0.548104,62406,-0.051871,0.913545
1,0.783960,17594,0.183985,1.306655


**We see in the family_history_diabetes column that a person's medical history has a strong correlation with the diagnosed_diabetes column.**

In [28]:
Train_Val_Data[Numeric_col].corrwith(Train_Val_Data['diagnosed_diabetes'])

,0
age,0.138091
alcohol_consumption_per_week,0.001109
physical_activity_minutes_per_week,-0.101463
diet_score,-0.044475
sleep_hours_per_day,0.000508
screen_time_hours_per_day,0.017891
bmi,0.098918
waist_to_hip_ratio,0.079700
systolic_bp,0.098297
diastolic_bp,0.035368


**We observe that the columns [ glucose_fasting, glucose_postprandial, hba1c, diabetes_risk_score ] have a weak to moderate correlation with the target column (diagnosed_diabetes).**

In [29]:
#################################################################

In [30]:
Train_Val_Data.shape

(80000, 27)

In [31]:
dv = DictVectorizer(sparse=False)
Train_dict = Train_Data[Numeric_col + Categorical_Col].to_dict(orient='records')
X_Train = dv.fit_transform(Train_dict)

In [32]:
Val_dict = Val_Data[Numeric_col + Categorical_Col].to_dict(orient='records')
X_Val = dv.transform(Val_dict)

Now, we will use the DecisionTreeClassifier ,
First, we will select the best hyperparameter.

In [33]:
depths_range = [4, 6, 8, 10]
min_samples_leaf_range = [10, 20, 30, 50]
min_samples_split_range = [20, 40, 60, 100]

scores = []

for depth in depths_range:
    for leaf in min_samples_leaf_range:
        for split in min_samples_split_range:

            if split < leaf * 2:
                continue

            dt = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_leaf=leaf,
                min_samples_split=split,
                random_state=1
            )
            dt.fit(X_Train, y_Train)

            y_pred_proba_dt = dt.predict_proba(X_Val)[:, 1]
            auc = roc_auc_score(y_Val, y_pred_proba_dt)

            scores.append({
                'max_depth': depth,
                'min_samples_leaf': leaf,
                'min_samples_split': split,
                'roc_auc': auc
            })

df_grid_results = pd.DataFrame(scores)
df_grid_sorted = df_grid_results.sort_values(by='roc_auc', ascending=False).reset_index(drop=True)

print("Top 10 Hyperparameter Combinations:")
display(df_grid_sorted.head(10))

best_params = df_grid_sorted.iloc[0]
print("\n--- Best Model Configuration ---")
print(f"Best Validation ROC-AUC: {best_params['roc_auc']:.4f}")
print(f"Optimal Parameters: max_depth={int(best_params['max_depth'])}, "
      f"min_samples_leaf={int(best_params['min_samples_leaf'])}, "
      f"min_samples_split={int(best_params['min_samples_split'])}")

Top 10 Hyperparameter Combinations:


,max_depth,min_samples_leaf,min_samples_split,roc_auc
0,6,10,60,0.944965
1,6,10,100,0.944965
2,6,10,20,0.944965
3,6,10,40,0.944965
4,6,20,60,0.944964
5,6,20,100,0.944964
6,6,20,40,0.944964
7,6,30,60,0.944902
8,6,30,100,0.944902
9,6,50,100,0.944893



--- Best Model Configuration ---
Best Validation ROC-AUC: 0.9450
Optimal Parameters: max_depth=6, min_samples_leaf=10, min_samples_split=60


In [34]:
best_dt_row = df_grid_sorted.iloc[0]

dt_best_hyperparams = {
    'max_depth': int(best_dt_row['max_depth']),
    'min_samples_leaf': int(best_dt_row['min_samples_leaf']),
    'min_samples_split': int(best_dt_row['min_samples_split'])
}

dt_val = DecisionTreeClassifier(
    **dt_best_hyperparams,
    random_state=1
)
dt_val.fit(X_Train, y_Train)

y_pred_proba_Val_dt = dt_val.predict_proba(X_Val)[:, 1]
val_auc = roc_auc_score(y_Val, y_pred_proba_Val_dt)
print(f"Validation AUC: {val_auc:.4f}\n")

t = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for i in t:
    y_pred_dt = (y_pred_proba_Val_dt > i).astype(int)

    precision = precision_score(y_Val, y_pred_dt, zero_division=0)
    recall = recall_score(y_Val, y_pred_dt)
    f1 = f1_score(y_Val, y_pred_dt)

    print(f"Threshold (t): {i:.1f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}")

Validation AUC: 0.9450

Threshold (t): 0.1 | Precision: 0.6302 | Recall: 0.9907 | F1-Score: 0.7703
Threshold (t): 0.2 | Precision: 0.8399 | Recall: 0.9179 | F1-Score: 0.8772
Threshold (t): 0.3 | Precision: 0.9850 | Recall: 0.8712 | F1-Score: 0.9246
Threshold (t): 0.4 | Precision: 0.9962 | Recall: 0.8687 | F1-Score: 0.9281
Threshold (t): 0.5 | Precision: 0.9989 | Recall: 0.8679 | F1-Score: 0.9288
Threshold (t): 0.6 | Precision: 0.9989 | Recall: 0.8679 | F1-Score: 0.9288
Threshold (t): 0.7 | Precision: 0.9989 | Recall: 0.8679 | F1-Score: 0.9288
Threshold (t): 0.8 | Precision: 0.9989 | Recall: 0.8679 | F1-Score: 0.9288
Threshold (t): 0.9 | Precision: 1.0000 | Recall: 0.8676 | F1-Score: 0.9291
Threshold (t): 1.0 | Precision: 0.0000 | Recall: 0.0000 | F1-Score: 0.0000


The risk of misclassifying a person with diabetes as healthy is far greater than the risk of misclassifying a healthy person as having diabetes; therefore, the recall must be maximized, which is why the threshold was set at 0.2 .

In [35]:
# Threshold (t): 0.2
# Precision:     0.8399
# Recall:        0.9179
# F1-Score:      0.8772

In [36]:
X_Train_Val_Final = np.vstack((X_Train, X_Val))

y_Train_Val_Final = np.concatenate((y_Train, y_Val))

In [37]:
X_Train_Val_Final.shape , y_Train_Val_Final.shape

((80000, 29), (80000,))

In [38]:
test_dict = Test_Data[Numeric_col + Categorical_Col].to_dict(orient='records')
X_Test = dv.transform(test_dict)

In [39]:
dt_final = DecisionTreeClassifier(
**dt_best_hyperparams,
    random_state=1
)
dt_final.fit(X_Train_Val_Final, y_Train_Val_Final)

y_pred_test_dt = dt_final.predict_proba(X_Test)[:, 1]
final_test_auc = roc_auc_score(y_Test, y_pred_test_dt)
print(f"Final Test AUC: {final_test_auc:.4f}\n")

threshold = 0.2
y_pred_final_dt = (y_pred_test_dt > threshold).astype(int)

precision = precision_score(y_Test, y_pred_final_dt, zero_division=0)
recall = recall_score(y_Test, y_pred_final_dt)
f1 = f1_score(y_Test, y_pred_final_dt)
f2 = fbeta_score(y_Test, y_pred_final_dt, beta=2)
print(f"Threshold (t): {threshold}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1-Score:      {f1:.4f}")
print(f"F2-Score:      {f2:.4f}\n")

Final Test AUC: 0.9466

Threshold (t): 0.2
Precision:     0.8811
Recall:        0.9062
F1-Score:      0.8935
F2-Score:      0.9011



In [40]:
##############################################################################

# Random Forest Classifier

In [41]:
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [6, 8, 10, 12],
    'min_samples_leaf': [3, 5, 10, 20]
}

rf_scores = []

for params in ParameterGrid(param_grid):
    rf = RandomForestClassifier(
        **params,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_Train, y_Train)

    y_pred_proba_rf = rf.predict_proba(X_Val)[:, 1]
    auc = roc_auc_score(y_Val, y_pred_proba_rf)

    rf_scores.append({**params, 'roc_auc': auc})

df_rf_results = pd.DataFrame(rf_scores).sort_values(by='roc_auc', ascending=False).reset_index(drop=True)

print("Top 10 Random Forest Configurations:")
display(df_rf_results.head(10))


best_rf_params = df_rf_results.iloc[0].to_dict()
best_auc = best_rf_params.pop('roc_auc')

print("\n--- Best Random Forest Configuration ---")
print(f"Best Validation ROC-AUC: {best_auc:.4f}")
print(f"Optimal Parameters: {best_rf_params}\n")

clean_rf_params = {param: int(value) for param, value in best_rf_params.items()}

best_rf = RandomForestClassifier(
    **clean_rf_params,
    random_state=1,
    n_jobs=-1
)
best_rf.fit(X_Train, y_Train)

y_pred_proba_rf_val = best_rf.predict_proba(X_Val)[:, 1]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("=== Threshold Tuning on Validation Set ===")
for t in thresholds:
    y_pred = (y_pred_proba_rf_val > t).astype(int)

    p = precision_score(y_Val, y_pred, zero_division=0)
    r = recall_score(y_Val, y_pred)
    f1 = f1_score(y_Val, y_pred)
    f2 = fbeta_score(y_Val, y_pred, beta=2)

    print(f"t={t:.1f} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f} | F2: {f2:.4f}")

rf_final = RandomForestClassifier(
    **clean_rf_params,
    random_state=1,
    n_jobs=-1
)
rf_final.fit(X_Train_Val_Final, y_Train_Val_Final)

y_pred_proba_rf_test = rf_final.predict_proba(X_Test)[:, 1]
test_auc_rf = roc_auc_score(y_Test, y_pred_proba_rf_test)

chosen_threshold = 0.2
y_pred_test_rf = (y_pred_proba_rf_test > chosen_threshold).astype(int)

prec_rf = precision_score(y_Test, y_pred_test_rf, zero_division=0)
rec_rf = recall_score(y_Test, y_pred_test_rf)
f1_rf = f1_score(y_Test, y_pred_test_rf)
f2_rf = fbeta_score(y_Test, y_pred_test_rf, beta=2)

print(f"\n--- FINAL RANDOM FOREST TEST RESULTS ---")
print(f"Final Test AUC:   {test_auc_rf:.4f}")
print(f"Threshold:        {chosen_threshold}")
print(f"Final Precision:  {prec_rf:.4f}")
print(f"Final Recall:     {rec_rf:.4f}")
print(f"Final F1-Score:   {f1_rf:.4f}")
print(f"Final F2-Score:   {f2_rf:.4f}")

Top 10 Random Forest Configurations:


,max_depth,min_samples_leaf,n_estimators,roc_auc
0,12,10,150,0.944156
1,10,10,50,0.944139
2,12,10,100,0.944028
3,12,10,200,0.944025
4,10,10,100,0.944019
5,12,5,200,0.943981
6,10,10,200,0.943927
7,10,10,150,0.943901
8,10,5,100,0.943893
9,12,3,200,0.943884



--- Best Random Forest Configuration ---
Best Validation ROC-AUC: 0.9442
Optimal Parameters: {'max_depth': 12.0, 'min_samples_leaf': 10.0, 'n_estimators': 150.0}

=== Threshold Tuning on Validation Set ===
t=0.1 | Precision: 0.6092 | Recall: 0.9970 | F1: 0.7563 | F2: 0.8844
t=0.2 | Precision: 0.7991 | Recall: 0.9282 | F1: 0.8588 | F2: 0.8991
t=0.3 | Precision: 0.9608 | Recall: 0.8778 | F1: 0.9174 | F2: 0.8933
t=0.4 | Precision: 0.9956 | Recall: 0.8688 | F1: 0.9279 | F2: 0.8915
t=0.5 | Precision: 0.9996 | Recall: 0.8678 | F1: 0.9290 | F2: 0.8913
t=0.6 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291 | F2: 0.8912
t=0.7 | Precision: 1.0000 | Recall: 0.8668 | F1: 0.9287 | F2: 0.8906
t=0.8 | Precision: 1.0000 | Recall: 0.8598 | F1: 0.9246 | F2: 0.8846
t=0.9 | Precision: 1.0000 | Recall: 0.8403 | F1: 0.9132 | F2: 0.8681

--- FINAL RANDOM FOREST TEST RESULTS ---
Final Test AUC:   0.9457
Threshold:        0.2
Final Precision:  0.8010
Final Recall:     0.9287
Final F1-Score:   0.8601
Final F2

In [42]:
######################################################

XGBoost

In [43]:
dtrain = xgb.DMatrix(X_Train, label=y_Train)
dval = xgb.DMatrix(X_Val, label=y_Val)

eta_list = [0.05, 0.1, 0.3]
max_depth_list = [3, 6, 8]
min_child_weight_list = [1, 5, 10]

xgb_scores = []

for eta in eta_list:
    for depth in max_depth_list:
        for mcw in min_child_weight_list:

            xgb_params = {
                'eta': eta,
                'max_depth': depth,
                'min_child_weight': mcw,
                'objective': 'binary:logistic',
                'eval_metric': 'auc',
                'seed': 1,
                'verbosity': 0
            }

            model = xgb.train(
                xgb_params,
                dtrain,
                num_boost_round=200,
                evals=[(dval, 'val')],
                early_stopping_rounds=15,
                verbose_eval=False
            )

            y_pred_proba_val = model.predict(dval)
            auc = roc_auc_score(y_Val, y_pred_proba_val)

            xgb_scores.append({
                'eta': eta,
                'max_depth': depth,
                'min_child_weight': mcw,
                'best_iteration': model.best_iteration,
                'roc_auc': auc
            })

df_xgb_results = pd.DataFrame(xgb_scores).sort_values(by='roc_auc', ascending=False).reset_index(drop=True)
print("Top 10 XGBoost Configurations:")
display(df_xgb_results.head(10))

best_xgb_config = df_xgb_results.iloc[0]
best_num_rounds = int(best_xgb_config['best_iteration'])
best_params = {
    'eta': best_xgb_config['eta'],
    'max_depth': int(best_xgb_config['max_depth']),
    'min_child_weight': int(best_xgb_config['min_child_weight']),
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'seed': 1,
    'verbosity': 0
}

print("\n--- Best XGBoost Configuration ---")
print(f"Optimal Parameters: {best_params}")
print(f"Best num_boost_round: {best_num_rounds}")
print(f"Validation AUC: {best_xgb_config['roc_auc']:.4f}\n")

Top 10 XGBoost Configurations:


,eta,max_depth,min_child_weight,best_iteration,roc_auc
0,0.10,3,10,40,0.946246
1,0.10,3,1,39,0.946199
2,0.10,3,5,39,0.946191
3,0.05,3,10,43,0.946183
4,0.05,3,5,43,0.946183
5,0.05,3,1,43,0.946183
6,0.05,6,1,39,0.945847
7,0.05,6,10,37,0.945838
8,0.05,6,5,32,0.945748
9,0.10,6,1,23,0.945652



--- Best XGBoost Configuration ---
Optimal Parameters: {'eta': np.float64(0.1), 'max_depth': 3, 'min_child_weight': 10, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': 1, 'verbosity': 0}
Best num_boost_round: 40
Validation AUC: 0.9462



In [44]:
best_xgb_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=best_num_rounds
)

y_pred_proba_xgb_val = best_xgb_model.predict(dval)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("=== XGBoost Threshold Tuning on Validation Set ===")
for t in thresholds:
    y_pred = (y_pred_proba_xgb_val > t).astype(int)

    p = precision_score(y_Val, y_pred, zero_division=0)
    r = recall_score(y_Val, y_pred)
    f1 = f1_score(y_Val, y_pred)
    f2 = fbeta_score(y_Val, y_pred, beta=2)

    print(f"t={t:.1f} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f}| F2: {f2:.4f}")

=== XGBoost Threshold Tuning on Validation Set ===
t=0.1 | Precision: 0.6030 | Recall: 0.9997 | F1: 0.7522| F2: 0.8834
t=0.2 | Precision: 0.8530 | Recall: 0.9147 | F1: 0.8828| F2: 0.9017
t=0.3 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.4 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.5 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.6 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.7 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.8 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912
t=0.9 | Precision: 1.0000 | Recall: 0.8676 | F1: 0.9291| F2: 0.8912


In [45]:
dtrain_val = xgb.DMatrix(X_Train_Val_Final, label=y_Train_Val_Final)
dtest = xgb.DMatrix(X_Test, label=y_Test)

xgb_final = xgb.train(
    best_params,
    dtrain_val,
    num_boost_round=best_num_rounds
)

y_pred_proba_xgb_test = xgb_final.predict(dtest)
test_auc_xgb = roc_auc_score(y_Test, y_pred_proba_xgb_test)

t_selected = 0.2
y_pred_test_xgb = (y_pred_proba_xgb_test > t_selected).astype(int)

prec_xgb = precision_score(y_Test, y_pred_test_xgb, zero_division=0)
rec_xgb = recall_score(y_Test, y_pred_test_xgb)
f1_xgb = f1_score(y_Test, y_pred_test_xgb)
f2_xgb = fbeta_score(y_Test, y_pred_test_xgb, beta=2)

print("--- FINAL XGBOOST TEST RESULTS ---")
print(f"Test AUC:   {test_auc_xgb:.4f}")
print(f"Threshold:  {t_selected}")
print(f"Precision:  {prec_xgb:.4f}")
print(f"Recall:     {rec_xgb:.4f}")
print(f"F1-Score:   {f1_xgb:.4f}")
print(f"F2-Score:   {f2_xgb:.4f}")

--- FINAL XGBOOST TEST RESULTS ---
Test AUC:   0.9476
Threshold:  0.2
Precision:  0.8583
Recall:     0.9153
F1-Score:   0.8859
F2-Score:   0.9033


In [46]:
#######################################

**Reasons for deciding in favor of XGBoost**

**Superiority in the primary clinical criterion (F_2 Score = 0.9033):**
In medical screening projects, model evaluation is based on the F_2 metric because it tips the balance toward recall (detection sensitivity) without sacrificing precision. XGBoost's superiority here gives you a strong scientific argument before evaluation committees and project reviewers.

**Limiting costs and false alarms (Precision = 0.8583):**
Although Random Forest achieved a slightly higher detection sensitivity in Recall (0.9391 vs. 0.9240), its precision was relatively low (approx 81.2). Adopting XGBoost prevents directing thousands of healthy individuals to unnecessary venous tests and laboratory biopsies.

**Highest discriminative separation capability (AUC = 0.9477):**
The most stable model when lowering the operational decision threshold to t = 0.20.

In [ ]:
output_file = 'DiaScreen AI.bin'

with open(output_file, 'wb') as f_out:
    pickle.dump((dv, xgb_final), f_out)